In [ ]:
import mpmath as mp
import numpy as np

In [ ]:
mp.hyp2f1(-1,-1,2,-.1)**2

In [ ]:
eta = .1
nb = .2
ns = .01
sumratio = eta*nb/(1-eta+nb)
hyperval = -eta/(1-eta)
args = (sumratio,hyperval)
rat1 = ns*(1-eta)/(1+ns)
rat2 = nb*(1-eta)/(1-eta+nb)
argsc = (rat1,rat2)
hyp2f1 = mp.memoize(mp.hyp2f1)

In [ ]:
def Base_Sum_Generator(m,n,*args):
    sumrat,hyperval = args
    def base_sum(k):
        binoms = mp.binomial(n+k,n)*mp.binomial(m+k,m)
        hyper = hyp2f1(-m,-n,k+1,hyperval)**2
        return binoms*hyper*sumrat**k
    return mp.memoize(base_sum)

In [ ]:
base_sum_inst = Base_Sum_Generator(10,10,*args)

In [ ]:
base_sum_inst(0)

In [ ]:
mp.nsum(base_sum_inst,[0,mp.inf],verbose=True)

In [ ]:
def K_Sum_Generator(m,n,*args):
    sumrat,hyperval = args
    def base_sum(k):
        binoms = mp.binomial(n+k,n)*mp.binomial(m+k,m)
        hyper = hyp2f1(-m,-n,k+1,hyperval)**2
        return k*binoms*hyper*sumrat**k
    return mp.memoize(base_sum)

In [ ]:
def F_Sum_Generator(m,n,*args):
    sumrat,hyperval = args
    def base_sum(k):
        binoms = mp.binomial(n+k,n)*mp.binomial(m+k,m)
        hyper = hyp2f1(-m,-n,k+1,hyperval)*hyp2f1(1-m,1-n,k+1,hyperval)
        return m*n*binoms*hyper*sumrat**k/(1+k)
    return mp.memoize(base_sum)

In [ ]:
k_sum_inst = Base_Sum_Generator(10,10,*args)

In [ ]:
mp.nsum(k_sum_inst,[0,mp.inf])

In [ ]:
f_sum_inst = Base_Sum_Generator(10,10,*args)

In [ ]:
def fisher_sum_gen(gen1,gen2,consts,args1,args2,argsb,argsc):
    def fisher_sum(m,n):
        inst1 = gen1(m,n,*args1)
        inst2 = gen2(m,n,*args2)
        instb = Base_Sum_Generator(m,n,*argsb)
        sumb = mp.nsum(instb,[0,mp.inf],method='d')
        sum1 = mp.nsum(inst1,[0,mp.inf],method='d')
        sum2 = mp.nsum(inst2,[0,mp.inf],method='d')
        return sum1*sum2/sumb*consts(m,n,*argsc)
    return fisher_sum

In [ ]:
def consts(m,n,*args):
    rat1,rat2 = args
    return rat1**m*rat2**n

In [ ]:
t_sum_inst = fisher_sum_gen(F_Sum_Generator,F_Sum_Generator,consts,args,args,args,argsc)

In [ ]:
mp.nsum(t_sum_inst,[0,mp.inf],[0,mp.inf],method='d')

Try to test timing

In [ ]:
timingargs = (f_sum_inst,[0,mp.inf])
#mp.nsum(k_sum_inst,[0,mp.inf])
def eval_setting(setting,n=10):
    timingkwargs = {'method': setting}
    #base = mp.timing(mp.nsum,*timingargs,**timingkwargs)
    time = 0
    #n = int(min(np.ceil(tot/base),1000))
    for i in range(n):
        time += mp.timing(mp.nsum,*timingargs,**timingkwargs)
    return time/n
rtime = eval_setting('r')
stime = eval_setting('s')
ltime = eval_setting('l')
etime = eval_setting('e')
dtime = eval_setting('d')
outtext = f"r:{rtime}  s:{stime}  l:{ltime}  e:{etime}  d:{dtime}"
print(outtext)

In [ ]:
timingargs2 = (t_sum_inst,[0,mp.inf],[0,mp.inf])
#mp.nsum(k_sum_inst,[0,mp.inf])
def eval_setting_overall(setting,n=10):
    timingkwargs = {'method': setting}
    time = 0
    for i in range(n):
        time += mp.timing(mp.nsum,*timingargs2,**timingkwargs)
    return time/n
rtime = eval_setting('r')
stime = eval_setting('s')
ltime = eval_setting('l')
etime = eval_setting('e')
dtime = eval_setting('d')
outtext = f"r:{rtime}  s:{stime}  l:{ltime}  e:{etime}  d:{dtime}"
print(outtext)

It looks like we have

fsums -> d

ksums -> d

basesums -> d

In [ ]:
def information_correction(eta,nb,ns):
    sumratio = eta*nb/(1-eta+nb)
    hyperval = -eta/(1-eta)
    args = (sumratio,hyperval)
    rat1 = ns*(1-eta)/(1+ns)
    rat2 = nb*(1-eta)/(1-eta+nb)
    argsc = (rat1,rat2)
    ff_sum_inst = fisher_sum_gen(F_Sum_Generator,F_Sum_Generator,consts,args,args,args,argsc)
    kk_sum_inst = fisher_sum_gen(K_Sum_Generator,K_Sum_Generator,consts,args,args,args,argsc)
    fk_sum_inst = fisher_sum_gen(F_Sum_Generator,K_Sum_Generator,consts,args,args,args,argsc)
    ff_sum = mp.nsum(ff_sum_inst,[0,mp.inf],[0,mp.inf],method='d')
    kk_sum = mp.nsum(kk_sum_inst,[0,mp.inf],[0,mp.inf],method='d')
    fk_sum = mp.nsum(fk_sum_inst,[0,mp.inf],[0,mp.inf],method='d')
    Ic = (1-eta)*(kk_sum*(1+nb)**2/(eta**2*(1-eta+nb)**2) +4*ff_sum/(1-eta)**2 -4*fk_sum*(1+nb)/(eta*(1-eta)*(1-eta+nb)))/((1-eta+nb)*(1+ns))
    return Ic

In [ ]:
information_correction(.1,.2,.01)

In [ ]:
def base_information(eta,nb,ns):
    oneder = 2 *ns*nb/((1-eta)*(1-eta+nb))
    expm = ns
    expn = eta*ns+nb
    expm2 = ns*(1+2*ns)
    expn2 = (eta*ns+ nb)*(1+2*(eta*ns+nb))
    expmn = ns*(eta*(1+2*ns)+nb)
    nodert1 = expm2/(1-eta)**2
    nodert2 = (expm + expmn)*2*nb/((1-eta)**2*(1-eta+nb))
    nodert3 = (1+2*expn + expn2)*nb**2/((1-eta)**2*(1-eta+nb)**2)
    noder = nodert1+nodert2+nodert3
    return -oneder -noder

In [ ]:
base_information(.1,.2,.01)

In [ ]:
def information(eta, nb, ns):
    return information_correction(eta,nb,ns) + base_information(eta,nb,ns)

In [ ]:
information(.1,.2,.01)

In [ ]:
def full_information(eta,nb,ns):
    return ns*(nb+1+ns*(nb+1-eta))/(eta*(nb+1-eta)*(nb+1+ns*(2*nb+1-eta)))

In [ ]:
full_information(.1,.2,.01)

In [ ]:
def basic_sum_gen(argsb,argsc):
    def fisher_sum(m,n):
        instb = Base_Sum_Generator(m,n,*argsb)
        sumb = mp.nsum(instb,[0,mp.inf],method='d')
        return sumb*consts(m,n,*argsc)
    return fisher_sum

In [ ]:
basic_sum_inst = basic_sum_gen(args,argsc)
base_sum = mp.nsum(basic_sum_inst,[0,mp.inf],[0,mp.inf],method='d')
res = base_sum/((1+nb)*(1+ns))

In [ ]:
res